In [11]:
tabla='ctcam10'
col_essi='fec_cit'
essi='essi_dat_cex001_etl'
col_dat='fec_cit'
dat='dat_cext001_essi'

In [3]:
from decouple import config
import decouple #eliminar en .py
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [721]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [722]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge
control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [6]:
config = decouple.AutoConfig(' ')
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [724]:
from datetime import datetime
from sqlalchemy import text


fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=2", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=2"

c1= text(query)
connection2.execute(c1)

2023-01-01


In [14]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

In [727]:
base1.columns

Index(['ori_cas', 'cod_cas', 'des_cas', 'cod_red', 'des_red', 'cit_num',
       'ori_pro', 'cas_pro', 'cod_are', 'des_are', 'cod_ser', 'des_ser',
       'cod_act', 'des_act', 'cod_sub', 'des_sub', 'cod_tdi', 'num_doc',
       'fec_cit', 'hor_ini', 'hor_ter', 'cod_tci', 'des_tci', 'cod_cci',
       'des_cci', 'cod_oto', 'des_oto', 'ord_cit', 'hor_cit', 'cit_rep',
       'des_rep', 'cod_mec', 'des_mec', 'cod_eci', 'des_eci', 'cod_enco',
       'des_enco', 'usu_cre', 'fec_cre', 'usu_mod', 'fec_mod', 'ip_mod',
       'fec_sol', 'ip_cre', 'mot_cit', 'pac_sec', 'usu_anu', 'fec_anu',
       'ip_anu', 'mod_anu', 'ctr_seg', 'cnv_esp'],
      dtype='object')

In [728]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127547 entries, 0 to 127546
Data columns (total 52 columns):
 #   Column    Non-Null Count   Dtype              
---  ------    --------------   -----              
 0   ori_cas   127547 non-null  object             
 1   cod_cas   127547 non-null  object             
 2   des_cas   127547 non-null  object             
 3   cod_red   127547 non-null  object             
 4   des_red   127547 non-null  object             
 5   cit_num   127547 non-null  float64            
 6   ori_pro   127547 non-null  object             
 7   cas_pro   127547 non-null  object             
 8   cod_are   127547 non-null  object             
 9   des_are   127547 non-null  object             
 10  cod_ser   127547 non-null  object             
 11  des_ser   127547 non-null  object             
 12  cod_act   127547 non-null  object             
 13  des_act   127547 non-null  object             
 14  cod_sub   127547 non-null  object             
 15  

In [729]:
base1.shape

(127547, 52)

In [730]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

In [731]:


tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_cit","dt_fecha":"fec_cit","dt_mes":"mes_cit","dt_dia":"dia_cit","dt_dia_sem":"dia_sem_cit","dt_sem":"sem_cit","dt_ano":"ano_cit"})
base1=pd.merge(base1,tiempo,how='left',on='fec_cit')

base1['fec_cre_temp']=base1['fec_cre']
base1['fec_mod_temp']=base1['fec_mod']
base1['fec_anu_temp']=base1['fec_anu']
base1['fec_sol_temp']=base1['fec_sol']

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_sol","dt_fecha":"fec_sol","dt_mes":"mes_sol","dt_dia":"dia_sol","dt_dia_sem":"dia_sem_sol","dt_sem":"sem_sol","dt_ano":"ano_sol"})
base1['fec_sol'] = base1['fec_sol'].astype(str).str[0:10]
base1["fec_sol"]=base1["fec_sol"].str.replace(' ', '', regex=True)
tiempo["fec_sol"]=tiempo["fec_sol"].astype(str).str.replace(' ', '', regex=True)
base1=pd.merge(base1,tiempo,how='left',on='fec_sol')


tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_cre","dt_fecha":"fec_cre","dt_mes":"mes_cre","dt_dia":"dia_cre","dt_dia_sem":"dia_sem_cre","dt_sem":"sem_cre","dt_ano":"ano_cre"})
base1['fec_cre'] = pd.to_datetime(base1['fec_cre']).dt.date
base1=pd.merge(base1,tiempo,how='left',on='fec_cre')

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_mod","dt_fecha":"fec_mod","dt_mes":"mes_mod","dt_dia":"dia_mod","dt_dia_sem":"dia_sem_mod","dt_sem":"sem_mod","dt_ano":"ano_mod"})
base1['fec_mod'] = pd.to_datetime(base1['fec_mod']).dt.date
base1=pd.merge(base1,tiempo,how='left',on='fec_mod')

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_anu","dt_fecha":"fec_anu","dt_mes":"mes_anu","dt_dia":"dia_anu","dt_dia_sem":"dia_sem_anu","dt_sem":"sem_anu","dt_ano":"ano_anu"})
#base1['fec_anu'] = pd.to_datetime(base1['fec_anu']).dt.date
base1=pd.merge(base1,tiempo,how='left',on='fec_anu')

base1['fec_cre'] = base1['fec_cre_temp']
base1['fec_mod'] = base1['fec_mod_temp']
base1['fec_anu'] = base1['fec_anu_temp']
base1['fec_sol'] = base1['fec_sol_temp']

base1 = base1.drop('fec_cre_temp', axis=1)
base1 = base1.drop('fec_mod_temp', axis=1)
base1 = base1.drop('fec_anu_temp', axis=1)
base1 = base1.drop('fec_sol_temp', axis=1)

In [ ]:
b

In [732]:
base1['ano_sol']

0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
          ..
127542   NaN
127543   NaN
127544   NaN
127545   NaN
127546   NaN
Name: ano_sol, Length: 127547, dtype: float64

In [733]:
control_a.append(base1.shape[0])

In [734]:
oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
merge=pd.merge(base1,oricas,how='left',on='ori_cas')
base1=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1.shape

(127547, 83)

In [735]:
log_falla('id_oricas', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [736]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_red is not null", con=connection2)
cas = cas.dropna()
merge=pd.merge(base1,cas,how='left',on='cod_cas')
base1=pd.merge(base1,cas,how='inner',on='cod_cas')
base1.shape

(127547, 83)

In [737]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [738]:
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
merge=pd.merge(base1,red,how='left',on='cod_red')
base1=pd.merge(base1,red,how='inner',on='cod_red')
base1.shape

(127547, 83)

In [739]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [740]:
oripro = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oripro=oripro.rename(columns={"ori_cod":"ori_pro"})
oripro=oripro.rename(columns={"id_oricas":"id_oripro"})
merge=pd.merge(base1,oripro,how='left',on='ori_pro')
base1=pd.merge(base1,oripro,how='inner',on='ori_pro')
base1.shape

(127547, 83)

In [741]:
log_falla('id_oripro', 'ori_pro', True)
log_control('dim_oricas')
base1=base1.drop('ori_pro',axis=1)

In [742]:
caspro = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_red is not null", con=connection2)
caspro = caspro.dropna()
caspro=caspro.rename(columns={"id_cas":"id_caspro"})
caspro=caspro.rename(columns={"cod_cas":"cas_pro"})
merge=pd.merge(base1,caspro,how='left',on='cas_pro')
base1=pd.merge(base1,caspro,how='left',on='cas_pro')
base1.shape

(127547, 83)

In [743]:
log_falla('id_caspro', 'cas_pro', True)
log_control('dim_cas')
base1=base1.drop('cas_pro',axis=1)

In [744]:
are = pd.read_sql_query(f"SELECT id_area,cod_are FROM dim_areas", con=connection2)
merge=pd.merge(base1,are,how='left',on='cod_are')
base1=pd.merge(base1,are,how='inner',on='cod_are')
base1.shape

(127547, 83)

In [745]:
log_falla('id_area', 'cod_are', True)
log_control('dim_areas')
base1=base1.drop('cod_are',axis=1)

In [746]:
serv= pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
merge=pd.merge(base1,serv,how='left',on='cod_ser')
base1=pd.merge(base1,serv,how='inner',on='cod_ser')
base1.shape

(127547, 83)

In [747]:
log_falla('id_serv', 'cod_ser', True)
log_control('dim_servicios')
base1=base1.drop('cod_ser',axis=1)

In [748]:
activi = pd.read_sql_query(f"SELECT id_activi,cod_act FROM dim_activi", con=connection2)
activi=activi.rename(columns={"id_activi":"id_acti"})
merge=pd.merge(base1,activi,how='left',on='cod_act')
base1=pd.merge(base1,activi,how='inner',on='cod_act')
base1.shape

(127547, 83)

In [749]:
log_falla('id_acti', 'cod_act', True)
log_control('dim_activi')

In [750]:
subacti = pd.read_sql_query(f"SELECT id_subacti,cod_sub,cod_act FROM dim_subacti", con=connection2)
subacti["KEY"]=subacti["cod_sub"]+subacti["cod_act"]
subacti=subacti.drop(["cod_sub",'cod_act'], axis=1)
base1["KEY"]=base1["cod_sub"].astype(str)+base1['cod_act'].astype(str)
base1["KEY"]=base1["KEY"].str.replace(' ', '', regex=True)
subacti["KEY"]=subacti["KEY"].str.replace(' ', '', regex=True)
merge = pd.merge(base1,subacti,how='left',on='KEY')
base1 = pd.merge(base1,subacti,how='inner',on='KEY')
base1.shape

(127547, 85)

In [751]:
base1.columns

Index(['des_cas', 'des_red', 'cit_num', 'des_are', 'des_ser', 'cod_act',
       'des_act', 'cod_sub', 'des_sub', 'cod_tdi', 'num_doc', 'fec_cit',
       'hor_ini', 'hor_ter', 'cod_tci', 'des_tci', 'cod_cci', 'des_cci',
       'cod_oto', 'des_oto', 'ord_cit', 'hor_cit', 'cit_rep', 'des_rep',
       'cod_mec', 'des_mec', 'cod_eci', 'des_eci', 'cod_enco', 'des_enco',
       'usu_cre', 'fec_cre', 'usu_mod', 'fec_mod', 'ip_mod', 'fec_sol',
       'ip_cre', 'mot_cit', 'pac_sec', 'usu_anu', 'fec_anu', 'ip_anu',
       'mod_anu', 'ctr_seg', 'cnv_esp', 'id_time_cit', 'mes_cit', 'dia_cit',
       'dia_sem_cit', 'sem_cit', 'ano_cit', 'id_time_sol', 'mes_sol',
       'dia_sol', 'dia_sem_sol', 'sem_sol', 'ano_sol', 'id_time_cre',
       'mes_cre', 'dia_cre', 'dia_sem_cre', 'sem_cre', 'ano_cre',
       'id_time_mod', 'mes_mod', 'dia_mod', 'dia_sem_mod', 'sem_mod',
       'ano_mod', 'id_time_anu', 'mes_anu', 'dia_anu', 'dia_sem_anu',
       'sem_anu', 'ano_anu', 'id_oricas', 'id_cas', 'id_red', 'id_o

In [752]:
base1.columns

Index(['des_cas', 'des_red', 'cit_num', 'des_are', 'des_ser', 'cod_act',
       'des_act', 'cod_sub', 'des_sub', 'cod_tdi', 'num_doc', 'fec_cit',
       'hor_ini', 'hor_ter', 'cod_tci', 'des_tci', 'cod_cci', 'des_cci',
       'cod_oto', 'des_oto', 'ord_cit', 'hor_cit', 'cit_rep', 'des_rep',
       'cod_mec', 'des_mec', 'cod_eci', 'des_eci', 'cod_enco', 'des_enco',
       'usu_cre', 'fec_cre', 'usu_mod', 'fec_mod', 'ip_mod', 'fec_sol',
       'ip_cre', 'mot_cit', 'pac_sec', 'usu_anu', 'fec_anu', 'ip_anu',
       'mod_anu', 'ctr_seg', 'cnv_esp', 'id_time_cit', 'mes_cit', 'dia_cit',
       'dia_sem_cit', 'sem_cit', 'ano_cit', 'id_time_sol', 'mes_sol',
       'dia_sol', 'dia_sem_sol', 'sem_sol', 'ano_sol', 'id_time_cre',
       'mes_cre', 'dia_cre', 'dia_sem_cre', 'sem_cre', 'ano_cre',
       'id_time_mod', 'mes_mod', 'dia_mod', 'dia_sem_mod', 'sem_mod',
       'ano_mod', 'id_time_anu', 'mes_anu', 'dia_anu', 'dia_sem_anu',
       'sem_anu', 'ano_anu', 'id_oricas', 'id_cas', 'id_red', 'id_o

In [753]:
log_falla('id_subacti', 'KEY', True)
log_control('dim_subacti')
base1=base1.drop('KEY', axis=1)
base1=base1.drop('cod_act',axis=1)

In [754]:
tipdoc = pd.read_sql_query(f"SELECT id_tipdoc,cod_tdo FROM dim_tipdoc", con=connection2)
tipdoc=tipdoc.rename(columns={"id_tipdoc":"id_tdi"})
tipdoc=tipdoc.rename(columns={"cod_tdo":"cod_tdi"})
merge=pd.merge(base1,tipdoc,how='left',on='cod_tdi')
base1=pd.merge(base1,tipdoc,how='inner',on='cod_tdi')
base1.shape

(127547, 84)

In [755]:
log_falla('id_tdi', 'cod_tdi', True)
log_control('dim_tipdoc')

In [756]:
base1

,des_cas,des_red,cit_num,des_are,des_ser,des_act,cod_sub,des_sub,cod_tdi,num_doc,...,id_oricas,id_cas,id_red,id_oripro,id_caspro,id_area,id_serv,id_acti,id_subacti,id_tdi
0,H.N. EDGARDO REBAGLIATI MARTINS,RED PRESTACIONAL REBAGLIATI ...,13307124.0,CONSULTA EXTERNA,UROLOGIA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,10289564,...,1,565,3,1,565,1,170,13,183,1
1,H.N. EDGARDO REBAGLIATI MARTINS,RED PRESTACIONAL REBAGLIATI ...,13305839.0,CONSULTA EXTERNA,UROLOGIA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,09933135,...,1,565,3,1,565,1,170,13,183,1
2,H.N. EDGARDO REBAGLIATI MARTINS,RED PRESTACIONAL REBAGLIATI ...,13485196.0,CONSULTA EXTERNA,UROLOGIA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,10289564,...,1,565,3,1,565,1,170,13,183,1
3,H.N. EDGARDO REBAGLIATI MARTINS,RED PRESTACIONAL REBAGLIATI ...,13308204.0,CONSULTA EXTERNA,UROLOGIA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,09933135,...,1,565,3,1,565,1,170,13,183,1
4,H.N. EDGARDO REBAGLIATI MARTINS,RED PRESTACIONAL REBAGLIATI ...,13306564.0,CONSULTA EXTERNA,UROLOGIA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,10289564,...,1,565,3,1,565,1,170,13,183,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127542,POL. CHINCHA,RED PRESTACIONAL REBAGLIATI ...,5474680.0,CONSULTA EXTERNA,ENDOCRINOLOGIA ...,ATENCION MEDICA AMBULATORIA ...,333,TELECONSULTA ...,2,000229329,...,1,125,3,1,125,1,14,13,759,2
127543,POL. CHINCHA,RED PRESTACIONAL REBAGLIATI ...,5474663.0,CONSULTA EXTERNA,ENDOCRINOLOGIA ...,ATENCION MEDICA AMBULATORIA ...,333,TELECONSULTA ...,2,000229329,...,1,125,3,1,125,1,14,13,759,2
127544,CAP III LUIS NEGREIROS VEGA,RED PRESTACIONAL SABOGAL ...,5642289.0,CONSULTA EXTERNA,MEDICINA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,002,APOYO PROCEDIMIENTOS EN HOSPITALIZACION ...,2,002872823,...,1,212,1,1,212,1,2,13,184,2
127545,POLICLINICO BAYOVAR,RED PRESTACIONAL ALMENARA ...,1238591.0,CONSULTA EXTERNA,CARDIOLOGIA ...,ATENCION PROCEDIMIENTOS ...,725,"EDUC.MED.CONT: COR.CLI.PAT,REV.BIB, AP.TEC.CIE...",2,107661762,...,2,563,2,2,563,1,12,18,344,2


In [757]:
base1['num_doc']

0         10289564  
1         09933135  
2         10289564  
3         09933135  
4         10289564  
             ...    
127542    000229329 
127543    000229329 
127544    002872823 
127545    107661762 
127546    107661762 
Name: num_doc, Length: 127547, dtype: object

In [758]:
personal = pd.read_sql_query(f"SELECT id_person,num_doc FROM dim_personal", con=connection2)
personal = personal.sort_values(by='id_person',ascending=False)
personal["num_doc"]=personal["num_doc"].str.replace(' ', '', regex=True)
personal=personal.drop_duplicates(subset="num_doc")
personal=personal.rename(columns={"id_person":"id_persona"})
base1["num_doc"]=base1["num_doc"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,personal,how='left',on='num_doc')
base1=pd.merge(base1,personal,how='inner',on='num_doc')

In [759]:
log_falla('num_doc', 'num_doc', True)
log_control('dim_personal')

In [760]:
numdoc = pd.read_sql_query(f"SELECT id_person, num_doc FROM dim_personal", con=connection2)
numdoc=numdoc.drop_duplicates(subset="num_doc")
numdoc=numdoc.rename(columns={"num_doc":"usu_cre"})
numdoc=numdoc.rename(columns={"id_person":"id_usucre"})
merge=pd.merge(base1,numdoc,how='left',on='usu_cre')
base1=pd.merge(base1,numdoc,how='left',on='usu_cre')
base1.shape

(127547, 86)

In [761]:
log_falla('id_usucre', 'usu_cre', True)
log_control('dim_personal')
base1=base1.drop('usu_cre',axis=1)

In [762]:
numdoc = pd.read_sql_query(f"SELECT id_person, num_doc FROM dim_personal", con=connection2)
numdoc=numdoc.drop_duplicates(subset="num_doc")
numdoc=numdoc.rename(columns={"num_doc":"usu_mod"})
numdoc=numdoc.rename(columns={"id_person":"id_usumod"})
merge=pd.merge(base1,numdoc,how='left',on='usu_mod')
base1=pd.merge(base1,numdoc,how='left',on='usu_mod')
base1.shape

(127547, 86)

In [763]:
log_falla('id_usumod', 'usu_mod', True)
log_control('dim_personal')
base1=base1.drop('usu_mod',axis=1)

In [764]:
numdoc = pd.read_sql_query(f"SELECT id_person, num_doc FROM dim_personal", con=connection2)
numdoc=numdoc.drop_duplicates(subset="num_doc")
numdoc=numdoc.rename(columns={"num_doc":"usu_anu"})
numdoc=numdoc.rename(columns={"id_person":"id_usuanu"})
merge=pd.merge(base1,numdoc,how='left',on='usu_anu')
base1=pd.merge(base1,numdoc,how='left',on='usu_anu')
base1.shape

(127547, 86)

In [765]:
log_falla('id_usuanu', 'usu_anu', True)
log_control('dim_personal')
base1=base1.drop('usu_anu',axis=1)

In [766]:
base1=base1.rename(columns={'hor_ter':'hor_fin'})

In [767]:
base1['hor_inix']=base1['hor_ini'].astype(str).str[10:30]
base1['hor_finx']=base1['hor_fin'].astype(str).str[10:30]
base1['hor_inix'] = base1['hor_inix'].str[0:6]
base1['hor_finx'] = base1['hor_finx'].str[0:6]
base1['hor_finx']=base1["hor_finx"].str.replace(' ', '', regex=True)
base1['hor_inix']=base1["hor_inix"].str.replace(' ', '', regex=True)
base1['turno']=base1['hor_inix'].astype(str)+"-"+base1['hor_finx'].astype(str)

In [768]:
turnos = pd.read_sql_query(f"SELECT id_turno,horas,minutos,destur,horario FROM dim_turnos", con=connection2)
turnos=turnos.rename(columns={"destur":"turno"})
turnos["turno"]=turnos["turno"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,turnos,how='left',on='turno')
base1=pd.merge(base1,turnos,how='inner',on='turno')


In [769]:
base1.shape

(127547, 92)

In [770]:
log_falla('id_turno', 'turno', True)
log_control('dim_turnos')
base1=base1.drop('turno',axis=1)

In [771]:
horario = pd.read_sql_query(f"SELECT id_horario, cod_hor FROM dim_horario", con=connection2)
horario = horario.rename(columns={'cod_hor':'horario'})
merge=pd.merge(base1,horario,how='left',on='horario')
base1=pd.merge(base1,horario,how='inner',on='horario')
base1.shape

(127547, 92)

In [772]:
log_falla('id_horario', 'horario', True)
log_control('dim_horario')
base1=base1.drop('horario',axis=1)

In [773]:
estsolcit = pd.read_sql_query(f"SELECT id_estsolcit,cod_esc FROM dim_cexestsolcit", con=connection2)
estsolcit = estsolcit.rename(columns={'id_estsolcit':'id_tci'})
estsolcit = estsolcit.rename(columns={'cod_esc':'cod_tci'})
merge=pd.merge(base1,estsolcit,how='left',on='cod_tci')
base1=pd.merge(base1,estsolcit,how='inner',on='cod_tci')
base1.shape

(127522, 92)

In [774]:
log_falla('id_tci', 'cod_tci', True)
log_control('dim_cexestsolcit')
base1=base1.drop('cod_tci', axis=1)

In [775]:
tipcita = pd.read_sql_query(f"SELECT id_tipocit,cod_tci FROM dim_tipcita", con=connection2)
tipcita = tipcita.rename(columns={'id_tipocit':'id_cci'})
tipcita = tipcita.rename(columns={'cod_tci':'cod_cci'})
merge=pd.merge(base1,tipcita,how='left',on='cod_cci')
base1=pd.merge(base1,tipcita,how='inner',on='cod_cci')
base1.shape

(127522, 92)

In [776]:
log_falla('id_cci', 'cod_cci', True)
log_control('dim_tipcita')
base1=base1.drop('cod_cci', axis=1)

In [777]:
tipemi = pd.read_sql_query(f"SELECT id_tipemi,cod_emi FROM dim_tipemi", con=connection2)
tipemi = tipemi.rename(columns={'id_tipemi':'id_otorga'})
tipemi = tipemi.rename(columns={'cod_emi':'cod_oto'})
merge=pd.merge(base1,tipemi,how='left',on='cod_oto')
base1=pd.merge(base1,tipemi,how='inner',on='cod_oto')
base1.shape

(127522, 92)

In [778]:
log_falla('id_otorga', 'cod_oto', True)
log_control('dim_tipemi')
base1=base1.drop('cod_oto', axis=1)

In [779]:
base1=base1.rename(columns={'cit_rep':'id_citrep'})

In [780]:
mec = pd.read_sql_query(f"SELECT id_moteli,cod_eli FROM dim_cexmotelicit", con=connection2)
mec = mec.rename(columns={'id_moteli':'id_mec'})
mec = mec.rename(columns={'cod_eli':'cod_mec'})
merge=pd.merge(base1,mec,how='left',on='cod_mec')
base1=pd.merge(base1,mec,how='left',on='cod_mec')
base1.shape

(127522, 92)

In [781]:
log_falla('id_mec', 'cod_mec', True)
log_control('dim_cexmotelicit')
base1=base1.drop('cod_mec', axis=1)

In [782]:
eci = pd.read_sql_query(f"SELECT id_estcit,cod_eci FROM dim_estcit", con=connection2)
eci = eci.rename(columns={'id_estcit':'id_eci'})
merge=pd.merge(base1,eci,how='left',on='cod_eci')
base1=pd.merge(base1,eci,how='inner',on='cod_eci')
base1.shape

(127522, 92)

In [783]:
log_falla('id_eci', 'cod_eci', True)
log_control('dim_estcit')
base1=base1.drop('cod_eci', axis=1)

In [784]:
enco = pd.read_sql_query(f"SELECT id_esteco,cod_eco FROM dim_cexcitoto", con=connection2)
enco = enco.rename(columns={'id_esteco':'id_enco'})
enco = enco.rename(columns={'cod_eco':'cod_enco'})
merge=pd.merge(base1,enco,how='left',on='cod_enco')
base1=pd.merge(base1,enco,how='inner',on='cod_enco')
base1.shape

(127522, 92)

In [ ]:
log_falla('id_enco', 'cod_enco', True)
log_control('dim_cexcitoto')
base1=base1.drop('cod_enco', axis=1)

In [786]:
pacsec = pd.read_sql_query(f"SELECT id_pacsec,per_sec FROM dim_pacsec", con=connection2)

pacsec=pacsec.rename(columns={"per_sec": "pac_sec"})
pacsec['pac_sec']=pacsec['pac_sec'].astype(str).str.strip()
pacsec["pac_sec"]=pacsec["pac_sec"].str.replace(' ', '', regex=True)
base1['pac_sec']=base1['pac_sec'].astype(str).str.strip()
base1["pac_sec"]=base1["pac_sec"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,pacsec,how='inner',on='pac_sec')
base1=pd.merge(base1,pacsec,how='left',on='pac_sec')
base1.shape

(127522, 91)

In [785]:
log_falla('id_pacsec', 'pac_sec', True)
base1=base1.drop('pac_sec', axis=1)
dim.append('dim_pacsec')
control_d.append(base1.shape[0])

In [787]:
base1['horfin']=base1['hor_finx']
base1['horini']=base1['hor_inix']
base1['horfin'] = base1['horfin'].str.strip()
base1['horfin'] = pd.to_datetime(base1['horfin'], format='%H:%M')
base1['horini'] = base1['horini'].str.strip()
base1['horini'] = pd.to_datetime(base1['horini'], format='%H:%M')
base1['diferencia_horas'] = base1['horfin'] - base1['horini']
base1['horas']=base1['diferencia_horas'].astype(str).str[-9:-6]

In [788]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [789]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [ ]:
query_count_before = f"SELECT COUNT(*) FROM {dat}"
cant_antes = connection2.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dat} antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla dat_cext001_essi antes de la inserción: 0


In [ ]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [ ]:
base.shape

(127522, 63)

In [ ]:
base

,cit_num,id_caspro,ano_cre,id_citrep,hor_fin,id_turno,id_eci,id_subacti,id_oripro,id_persona,...,mes_mod,id_otorga,hor_cit,id_area,id_enco,ano_anu,id_time_mod,dia_cre,dia_anu,sem_mod
0,13485196.0,565,2023.0,0,0001-01-01 20:00:00-04:56:16,17,2,183,1,3846629,...,5.0,1,0001-01-01 16:45:00-04:56:16,1,4,NaN,20230517.0,17.0,NaN,1.0
1,13306564.0,565,2023.0,0,0001-01-01 20:00:00-04:56:16,17,2,183,1,3846629,...,4.0,1,0001-01-01 14:00:00-04:56:16,1,4,NaN,20230419.0,19.0,NaN,1.0
2,13309947.0,565,2023.0,0,0001-01-01 20:00:00-04:56:16,17,2,183,1,3846629,...,4.0,1,0001-01-01 15:45:00-04:56:16,1,4,NaN,20230419.0,19.0,NaN,1.0
3,13309558.0,565,2023.0,0,0001-01-01 20:00:00-04:56:16,17,2,183,1,3846629,...,4.0,1,0001-01-01 14:45:00-04:56:16,1,4,NaN,20230419.0,19.0,NaN,1.0
4,13310021.0,565,2023.0,0,0001-01-01 20:00:00-04:56:16,17,2,183,1,3846629,...,4.0,1,0001-01-01 16:30:00-04:56:16,1,4,NaN,20230419.0,19.0,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127517,6904394.0,560,2023.0,0,0001-01-01 16:00:00-04:56:16,31,3,183,1,3771629,...,5.0,1,0001-01-01 12:36:00-04:56:16,1,4,NaN,20230520.0,20.0,NaN,1.0
127518,5605519.0,212,2023.0,0,0001-01-01 11:00:00-04:56:16,5,3,381,1,3882707,...,5.0,1,0001-01-01 09:00:00-04:56:16,1,4,NaN,20230509.0,9.0,NaN,1.0
127519,1957091.0,272,2023.0,0,0001-01-01 11:00:00-04:56:16,5,3,183,1,3809343,...,5.0,2,0001-01-01 07:36:00-04:56:16,1,4,NaN,20230522.0,22.0,NaN,1.0
127520,876057.0,111,2023.0,0,0001-01-01 11:00:00-04:56:16,5,3,183,1,3848545,...,5.0,2,0001-01-01 09:24:00-04:56:16,1,4,NaN,20230530.0,30.0,NaN,1.0


In [ ]:

base.to_sql(name=f'{dat}', con=engine2, if_exists='append', index=False,chunksize=5000)

25522

In [ ]:

query_count_after = f"SELECT COUNT(*) FROM {dat}"
cant_despues = connection2.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {dat} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")


Cantidad de filas en la tabla dat_cext001_essi después de la inserción: 127522
La cantidad de filas insertadas fue de: 127522


In [ ]:
#base=base.sort_values("fec_pro",ascending=False)
#fecha_fin= base.iloc[6, 1]
fecha_fin = "2024-12-31"

In [ ]:
proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=2", con=connection2)
proceso = proceso.iloc[0, 0]
cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=2", con=connection2)
cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")


In [ ]:
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado

nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']

tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [ ]:
base1

,des_cas,des_red,cit_num,des_are,des_ser,des_act,cod_sub,des_sub,cod_tdi,num_doc,...,id_horario,id_tci,id_cci,id_otorga,id_mec,id_eci,id_enco,horfin,horini,diferencia_horas
0,H.N. EDGARDO REBAGLIATI MARTINS,RED PRESTACIONAL REBAGLIATI ...,13485196.0,CONSULTA EXTERNA,UROLOGIA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,10289564,...,2,3,1,1,NaN,2,4,1900-01-01 20:00:00,1900-01-01 14:00:00,0 days 06:00:00
1,H.N. EDGARDO REBAGLIATI MARTINS,RED PRESTACIONAL REBAGLIATI ...,13306564.0,CONSULTA EXTERNA,UROLOGIA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,10289564,...,2,3,1,1,NaN,2,4,1900-01-01 20:00:00,1900-01-01 14:00:00,0 days 06:00:00
2,H.N. EDGARDO REBAGLIATI MARTINS,RED PRESTACIONAL REBAGLIATI ...,13309947.0,CONSULTA EXTERNA,UROLOGIA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,10289564,...,2,3,1,1,NaN,2,4,1900-01-01 20:00:00,1900-01-01 14:00:00,0 days 06:00:00
3,H.N. EDGARDO REBAGLIATI MARTINS,RED PRESTACIONAL REBAGLIATI ...,13309558.0,CONSULTA EXTERNA,UROLOGIA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,10289564,...,2,3,1,1,NaN,2,4,1900-01-01 20:00:00,1900-01-01 14:00:00,0 days 06:00:00
4,H.N. EDGARDO REBAGLIATI MARTINS,RED PRESTACIONAL REBAGLIATI ...,13310021.0,CONSULTA EXTERNA,UROLOGIA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,10289564,...,2,3,1,1,NaN,2,4,1900-01-01 20:00:00,1900-01-01 14:00:00,0 days 06:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127517,H.III HOSPITAL DE EMERGENCIAS GRAU,RED PRESTACIONAL ALMENARA ...,6904394.0,CONSULTA EXTERNA,GASTROENTEROLOGIA ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,06999980,...,2,2,1,1,NaN,3,4,1900-01-01 16:00:00,1900-01-01 12:00:00,0 days 04:00:00
127518,CAP III LUIS NEGREIROS VEGA,RED PRESTACIONAL SABOGAL ...,5605519.0,CONSULTA EXTERNA,NUTRICION ...,ATENCION NO MEDICA AMBULATORIA ...,050,CATETERISMO CON PRUEBAS DE VASOREACTIVIDAD PUL...,1,40222429,...,1,2,1,1,NaN,3,4,1900-01-01 11:00:00,1900-01-01 07:00:00,0 days 04:00:00
127519,H.II TARAPOTO,MICRORED ASISTENCIAL TARAPOTO ...,1957091.0,CONSULTA EXTERNA,OFTALMOLOGIA ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,40413158,...,1,2,1,2,NaN,3,4,1900-01-01 11:00:00,1900-01-01 07:00:00,0 days 04:00:00
127520,CAP III IQUITOS,RED ASISTENCIAL LORETO ...,876057.0,CONSULTA EXTERNA,MEDICINA GENERAL ...,ATENCION MEDICA AMBULATORIA ...,001,CONSULTA MEDICA ...,1,41017228,...,1,2,1,2,NaN,3,4,1900-01-01 11:00:00,1900-01-01 07:00:00,0 days 04:00:00


In [ ]:
tabla_logs

,codigo,proceso,dat,fecha_ejecucion,hora_inicio,hora_fin,dim,fecha_ini,fecha_ter,esperado,obtenido,faltante,falla,id_afectado
0,2.0,CONSULTA EXTERNA ...,dat_cext001_essi,2023-06-05,12:11,12:14,dim_oricas,2023-08-01,2024-12-31,127547,127547,0,0,id_oricas
1,2.0,CONSULTA EXTERNA ...,dat_cext001_essi,2023-06-05,12:11,12:14,dim_cas,2023-08-01,2024-12-31,127547,127547,0,0,id_cas
2,2.0,CONSULTA EXTERNA ...,dat_cext001_essi,2023-06-05,12:11,12:14,dim_red,2023-08-01,2024-12-31,127547,127547,0,0,id_red
3,2.0,CONSULTA EXTERNA ...,dat_cext001_essi,2023-06-05,12:11,12:14,dim_oricas,2023-08-01,2024-12-31,127547,127547,0,0,id_oripro
4,2.0,CONSULTA EXTERNA ...,dat_cext001_essi,2023-06-05,12:11,12:14,dim_cas,2023-08-01,2024-12-31,127547,127547,0,0,id_caspro
5,2.0,CONSULTA EXTERNA ...,dat_cext001_essi,2023-06-05,12:11,12:14,dim_areas,2023-08-01,2024-12-31,127547,127547,0,0,id_area
6,2.0,CONSULTA EXTERNA ...,dat_cext001_essi,2023-06-05,12:11,12:14,dim_servicios,2023-08-01,2024-12-31,127547,127547,0,0,id_serv
7,2.0,CONSULTA EXTERNA ...,dat_cext001_essi,2023-06-05,12:11,12:14,dim_activi,2023-08-01,2024-12-31,127547,127547,0,0,id_acti
8,2.0,CONSULTA EXTERNA ...,dat_cext001_essi,2023-06-05,12:11,12:14,dim_subacti,2023-08-01,2024-12-31,127547,127547,0,0,id_subacti
9,2.0,CONSULTA EXTERNA ...,dat_cext001_essi,2023-06-05,12:11,12:14,dim_tipdoc,2023-08-01,2024-12-31,127547,127547,0,0,id_tdi


In [ ]:
tabla_logs.to_sql(name=f'logs', con=connection3, if_exists='append', index=False,chunksize=10000)

22

In [ ]:
connection1.close()
connection2.close()
connection3.close()
